# MCP Server Registry, Discovery, Trust, and Enterprise Gateways

Build and test an enterprise trust path from public discovery metadata to an evidence-backed registry record, a policy-enforcing gateway, an independently authorizing MCP server, and a verified effect. The lab is deterministic, offline, and uses the official Python MCP SDK in memory.

## Learning objectives

By the end you can separate discovery from admission, inspect a signed tenant-scoped trust record, detect registry and runtime drift, explain safe gateway token exchange, verify server-side authorization, test revocation, and interpret both safety and usability metrics.

In [ ]:
from dataclasses import asdict, replace
from datetime import timedelta
import json
import runpy
from pathlib import Path

import jwt

lab = runpy.run_path(Path('lab.py'))
NOW = lab['utc']('2026-10-07T17:30:00Z')
print('Loaded Course 17 lab at', NOW.isoformat())

## 1. Trust boundary: metadata proposes; enterprise policy disposes

A public Registry entry can help locate a logical server, package, repository, or remote endpoint. Publisher namespace authentication does not prove safe bytes, reviewed capabilities, the identity of a running workload, tenant approval, or user authorization. The private record must bind those facts explicitly.

In [ ]:
env = await lab['reviewed_environment'](NOW)
public_metadata = asdict(env.metadata)
public_metadata['registry_updated_at'] = public_metadata['registry_updated_at'].isoformat()
public_metadata

**Checkpoint:** identify which fields above came from the publisher. Which additional evidence would you need before executing the package or sending it customer data?

## 2. Evidence-backed enterprise admission

Admission intersects public metadata with independently reviewed artifact, source, provenance, SBOM, vulnerability, endpoint, workload, contract, ownership, data-classification, isolation, and capability evidence. The resulting approval is scoped to one tenant and environment and expires.

In [ ]:
record_summary = {
    'scope': [env.record.tenant_id, env.record.environment],
    'server': [env.record.server_name, env.record.server_version],
    'status': env.record.status,
    'artifact_digest': env.record.artifact_digest,
    'contract_digest': env.record.contract_digest,
    'endpoint': env.record.endpoint,
    'workload_identity': env.record.workload_identity,
    'expires_at': env.record.expires_at.isoformat(),
    'record_digest': env.record.record_digest,
}
record_summary

In [ ]:
[
    {
        'enterprise_capability': grant.exposed_name,
        'mcp_tool': grant.server_tool,
        'application_permission': grant.required_permission,
        'downstream_scope': grant.downstream_scope,
        'risk': grant.risk,
        'rate_limit': grant.max_calls_per_minute,
    }
    for grant in env.record.grants
]

## 3. Signed records, ordered snapshots, and atomic caches

A snapshot is a complete authorization state. Consumers verify it off-path, reject rollback or same-epoch equivocation, and replace local state atomically. A production consumer must persist the highest accepted epoch across restarts; an in-memory counter alone permits rollback after a crash.

In [ ]:
before = (env.cache.epoch, env.cache.snapshot_digest)
tampered = replace(env.snapshot, epoch=env.snapshot.epoch + 99)
try:
    env.cache.sync(tampered, NOW)
    tamper_result = 'UNSAFE: accepted'
except lab['RegistryDenied'] as error:
    tamper_result = error.code
after = (env.cache.epoch, env.cache.snapshot_digest)
assert before == after
{'decision': tamper_result, 'cache_unchanged': before == after}

## 4. The gateway checks the live deployment

The gateway resolves the signed record using the authenticated tenant, then compares the exact server version, endpoint, workload identity, artifact digest, and live MCP contract. It validates arguments, permission, rate limit, and protocol before issuing narrow downstream authority.

In [ ]:
live_contract_digest, live_tools = await lab['inspect_contract'](env.target.mcp)
assert live_contract_digest == env.record.contract_digest
{'tools': live_tools, 'contract_digest': live_contract_digest, 'matches_review': True}

## 5. Caller claims are not authenticated context

The request deliberately claims tenant `globex` and supplies an attacker-controlled bearer value. The gateway uses the principal established by the application session (`acme`) and never forwards the caller token.

In [ ]:
call = lab['request']()
{
    'caller_claimed_tenant': call.claimed_tenant,
    'caller_supplied_token': call.client_authorization,
    'authenticated_tenant': env.principal.tenant_id,
    'authenticated_permissions': sorted(env.principal.permissions),
}

In [ ]:
approved = await env.gateway.handle(call, env.principal, env.target, NOW)
assert approved.decision == lab['Decision'].ALLOW
{
    'decision': approved.decision,
    'reason': approved.reason_code,
    'ticket': approved.response.ticket.model_dump() if approved.response else None,
    'child_token_issued': approved.downstream_jti is not None,
    'effect_count': env.target.calls['ticket.read'],
}

In [ ]:
audit = asdict(env.gateway.audit[-1])
audit['occurred_at'] = audit['occurred_at'].isoformat()
audit

## 6. Drift and tenant confusion must fail before effects

A familiar name is insufficient. These cases independently change tenant context, bytes, network target, workload identity, and the actual SDK-returned tool contract. Every unsafe case must deny with zero effects.

In [ ]:
drift_ids = {
    'wrong-tenant-record', 'artifact-drift', 'endpoint-swap',
    'workload-identity-drift', 'contract-drift'
}
drift_results = [
    await lab['run_case'](case)
    for case in lab['CASES']
    if case.case_id in drift_ids
]
assert all(not item.actual_allow and item.effect_count == 0 for item in drift_results)
[asdict(item) for item in drift_results]

## 7. The MCP server remains a resource server

The gateway mints a short-lived token bound to the exact audience, caller, tenant, scope, logical server, and registry record. The MCP tool independently validates that authority and enforces ticket ownership. This is an offline JWT model, not a production authorization server or token exchange.

In [ ]:
grant = env.record.grants[0]
child = env.broker.issue(env.principal, env.record, grant, NOW, 'req-inspect-token')
claims = jwt.decode(child, options={'verify_signature': False})
{key: claims[key] for key in ('iss', 'aud', 'sub', 'tenant', 'scope', 'server', 'registry_record', 'iat', 'exp', 'jti')}

In [ ]:
bypass_allowed = await lab['direct_call_without_gateway'](env.target, NOW)
assert bypass_allowed is False
{'direct_call_allowed': bypass_allowed, 'invariant': 'server authorization is independent'}

## 8. Revocation is a distributed operation

The authoritative record changes first. Signed state then reaches the host and resource server, and authority bound to the previous record is revoked. The outcome oracle is the ticket-read effect counter, not the denial text.

In [ ]:
revocation = await lab['revocation_exercise']()
assert revocation['before_revocation'] == lab['Decision'].ALLOW
assert revocation['after_revocation'] == lab['Decision'].DENY
assert revocation['effect_count'] == 1
revocation

## 9. Evaluate safety and usefulness together

A gateway that denies everything has no unsafe allows but is not useful. The campaign contains a labelled valid task plus unsafe tenant, permission, capability, drift, stale-cache, revocation, input, and resource-ownership cases.

In [ ]:
campaign = await lab['run_campaign']()
assert all(item.expected_allow == item.actual_allow for item in campaign.results)
{
    'metrics': campaign.metrics(),
    'cases': [asdict(item) for item in campaign.results],
}

**Checkpoint:** explain why `unsafe_allow_rate` uses labelled unsafe cases as its denominator, while `safe_task_completion_rate` uses labelled valid cases. What additional uncertainty would you report for a larger sampled campaign?

## 10. Map the simulation to production components

| Lab component | Production category | Examples |
|---|---|---|
| `PublicServerMetadata` | discovery catalog | official MCP Registry, private Registry-compatible catalog, Azure API Center |
| `TrustRegistry` | authoritative enterprise control plane | governed database/service with protected signing and audit |
| record evidence digests | supply-chain assurance | OCI/package digests, Cosign, SLSA provenance, CycloneDX/SPDX |
| `HostTrustCache` | distributed policy state | signed snapshot or policy-bundle client |
| `EnterpriseGateway` | proxy enforcement | Envoy + OPA, Docker MCP Gateway, Azure API Management, product gateways |
| workload string comparison | workload authentication | SPIFFE/SPIRE X.509-SVID, mesh/cloud workload identity |
| `TokenBroker` | delegated authority | OAuth authorization server and standards-compliant exchange/issuance |
| audit list | telemetry and evidence | OpenTelemetry Collector, governed audit store, SIEM |

## Production design exercise

Choose one MCP server used by your organization. Draft a record that names the exact version, package and digest, source revision, provenance/SBOM/scan evidence, complete contract digest, endpoint/resource identifier, workload identity, tenant/environment, owner, incident contact, data class, isolation profile, capability-to-permission/scope mappings, review expiry, and revocation procedure. Then draw every bypass route and identify which resource independently authorizes the final effect.

## Advanced implementation exercise

Replace one simulated boundary: (a) asymmetric registry signing with key rotation, (b) Envoy external authorization plus OPA, (c) SPIRE workload identity, or (d) a test authorization server issuing audience-bound child tokens. Add integration tests for outage, rollback, wrong issuer/audience, clock skew, bypass, and revocation. Do not claim the boundary is covered until an independent observer verifies the expected denial or effect.

## Reflection

1. What does official Registry namespace verification establish, and what remains unverified?
2. Why must the record bind both endpoint and workload identity?
3. How would you preserve the last accepted epoch across host restart?
4. When, if ever, may a host use a last-known-good registry snapshot during an outage?
5. Why can a gateway narrow authority but not replace resource ownership checks?
6. Which systems must acknowledge an emergency revocation before you declare containment?

## Authoritative references

- [MCP 2026-07-28 release](https://blog.modelcontextprotocol.io/posts/2026-07-28/)
- [Official MCP Registry documentation](https://github.com/modelcontextprotocol/registry/tree/main/docs)
- [Official Registry API](https://github.com/modelcontextprotocol/registry/blob/main/docs/reference/api/official-registry-api.md)
- [RFC 9728: OAuth Protected Resource Metadata](https://datatracker.ietf.org/doc/html/rfc9728)
- [Enterprise-Managed Authorization](https://blog.modelcontextprotocol.io/posts/enterprise-managed-auth/)
- [SPIFFE Workload API](https://spiffe.io/docs/latest/spiffe-specs/spiffe_workload_api/)
- [Envoy external authorization](https://www.envoyproxy.io/docs/envoy/latest/intro/arch_overview/security/ext_authz_filter)
- [Docker MCP Gateway](https://docs.docker.com/ai/mcp-catalog-and-toolkit/mcp-gateway/)
- [Azure API Center MCP registry](https://learn.microsoft.com/en-us/azure/api-center/register-discover-mcp-server)